# データセット探索

読み込んだデータの件数・分布・キーポイント座標の様子を確認する。

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from tennis_pose.dataset import TennisPoseDataset, CLASS_NAMES

## 1. データ読み込み

In [ ]:
dataset = TennisPoseDataset('../data', normalize=False)
print('データ数:', len(dataset))
print('特徴量の形状:', dataset.features.shape)
print('ラベルの種類:', set(dataset.labels.tolist()))

## 2. クラスごとのサンプル数

In [ ]:
counts = [(name, (dataset.labels == i).sum()) for i, name in enumerate(CLASS_NAMES)]
for name, count in counts:
    print(f'  {name}: {count}サンプル')

plt.figure(figsize=(7, 4))
plt.bar([c[0] for c in counts], [c[1] for c in counts], color='steelblue')
plt.title('クラスごとのサンプル数')
plt.ylabel('サンプル数')
plt.tight_layout()
plt.show()

## 3. 特徴量の分布（x座標・y座標）

In [ ]:
# x座標（偶数インデックス）とy座標（奇数インデックス）に分ける
x_coords = dataset.features[:, 0::2]  # shape: (2000, 18)
y_coords = dataset.features[:, 1::2]  # shape: (2000, 18)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(x_coords.flatten(), bins=50, color='steelblue', alpha=0.7)
axes[0].set_title('x座標の分布')
axes[0].set_xlabel('ピクセル値')

axes[1].hist(y_coords.flatten(), bins=50, color='tomato', alpha=0.7)
axes[1].set_title('y座標の分布')
axes[1].set_xlabel('ピクセル値')

plt.tight_layout()
plt.show()

## 4. クラスごとの関節位置（平均座標）

In [ ]:
JOINT_NAMES = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle', 'neck'
]

fig, axes = plt.subplots(1, 4, figsize=(16, 6))

for idx, (name, ax) in enumerate(zip(CLASS_NAMES, axes)):
    mask = dataset.labels == idx
    mean_xy = dataset.features[mask].mean(axis=0)  # (36,)
    mean_x = mean_xy[0::2]  # (18,)
    mean_y = mean_xy[1::2]  # (18,)

    ax.scatter(mean_x, -mean_y, s=60, color='tomato', zorder=3)  # y軸を反転（画像座標系）
    for i, joint in enumerate(JOINT_NAMES):
        ax.annotate(joint, (mean_x[i], -mean_y[i]), fontsize=5, ha='left')
    ax.set_title(name)
    ax.set_aspect('equal')
    ax.invert_yaxis()

plt.suptitle('クラスごとの平均関節位置')
plt.tight_layout()
plt.show()